In [ ]:
from cube import Cube
from solvers import SuperSolver
from cube_nn import CubeValueResNet
import torch

# Load NN value function
net = CubeValueResNet()
net.load_state_dict(torch.load(f'temp_models/resnet2.2/cube_value_resnet_iter_500.pth'))

# Move model to CUDA if available
device = 'cuda' if torch.cuda.is_available() else 'cpu'
net = net.to(device)
print(f'Using device: {device}')

solver = SuperSolver(net.as_value_function(), weight=0.25, noise=0.0,
                            max_moves=40, max_queue_size=1000000, t_max=60, max_restarts=2, batch_size=25, seed=42, dataset_moves=5)

/tmp/ipykernel_298119/667227794.py:8: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  net.load_state_dict(torch.load(f'temp_models/resnet2.2/cube_value_resnet_iter_500.pth'))


Using device: cuda
Loaded full optimal value dataset from datasets/full/full_5.pt.
Loaded full optimal value dataset from datasets/full/full_5.pt.


Traceback (most recent call last):
  File "/home/siro/Workspace/RubicksRL/venv/lib/python3.10/site-packages/matplotlib/cbook.py", line 361, in process
    func(*args, **kwargs)
  File "/home/siro/Workspace/RubicksRL/venv/lib/python3.10/site-packages/matplotlib/animation.py", line 932, in _start
    self.event_source.add_callback(self._step)
AttributeError: 'NoneType' object has no attribute 'add_callback'
Traceback (most recent call last):
  File "/home/siro/Workspace/RubicksRL/venv/lib/python3.10/site-packages/matplotlib/cbook.py", line 361, in process
    func(*args, **kwargs)
  File "/home/siro/Workspace/RubicksRL/venv/lib/python3.10/site-packages/matplotlib/animation.py", line 932, in _start
    self.event_source.add_callback(self._step)
AttributeError: 'NoneType' object has no attribute 'add_callback'
Traceback (most recent call last):
  File "/home/siro/Workspace/RubicksRL/venv/lib/python3.10/site-packages/matplotlib/cbook.py", line 361, in process
    func(*args, **kwargs)
  Fil

In [2]:
cube = Cube(n_scramble_moves=50, scramble_seed=0)

solution = solver(cube)
if solution is None:
    print('No solution found within constraints.')
else:
    print(f'Solution found with {len(solution)} moves: {solution}')

Solution found with 31 moves: [F, D, L, B2, L, D2, L, F2, D, L2, B, U', L', R, B, R', B', D', R2, B', R', D2, F', D2, B, R2, F', U2, F, U2, B']


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

initial_cube = cube.copy()

# Generate animation of the solution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 8), subplot_kw={'projection': '3d'})

def update(frame):
    """Update function for animation."""
    # Clear both axes
    ax1.clear()
    ax2.clear()
    
    # Apply moves up to current frame
    current_cube = initial_cube.copy()
    if frame > 0:
        current_cube.move(solution[:frame])
    
    # Draw front view
    current_cube.plot_3d(elev=30, azim=45, ax=ax1, show=False)
    if frame < len(solution):
        ax1.set_title(f'Front View - Move: {solution[frame]} ({frame+1}/{len(solution)})', fontsize=14, fontweight='bold')
    else:
        ax1.set_title(f'Front View - Solved! ({frame}/{len(solution)})', fontsize=14, fontweight='bold')
    # Draw back view
    current_cube.plot_3d(elev=30, azim=225, ax=ax2, show=False)
    ax2.set_title(f'Back View', fontsize=14, fontweight='bold')
    
    return ax1, ax2

# Create animation
anim = FuncAnimation(fig, update, frames=len(solution)+1, interval=500, blit=False, repeat=True)
plt.tight_layout()
plt.close()  # Prevent static display

# save animation as gif
anim.save('figs/cube_solution_animation.gif', writer='pillow', fps=2)

# Display animation
HTML(anim.to_jshtml())
